# A2 — P1_causal

**Shuffle all the rows, take 80% to train and 20% to test, and use the proper time-directed graph. The 'random split, good graph' arm.**

| | |
|---|---|
| arm | `P1_causal` — one of the four arms protocol v2 retrains |
| epochs | **200**, the canonical Section 5.1 budget (the old protocol study used 75) |
| graph | causal (`build_temporal_edges`, `assert_causal` on) |
| folds | 1 — random 80/20 over the same rows |
| estimated | ~0.8 h |
| writes to | `protocol_v2/results/arms/P1_causal/` and nowhere else |

### P3_rolling is not trained here, or anywhere

`P3_rolling` is the canonical Section 5.1 run. Protocol v2 reuses it exactly as it was
trained in August and **never** retrains or overwrites it. This notebook cannot train it:
`guards.assert_trainable` refuses the name, `config.ARM_SPEC` has no specification for it,
and `guards.assert_writable` refuses every path outside `protocol_v2/results/`. The
canonical directory's checksum is verified at the start and at the end of this notebook.

Do not confuse **`P3_acausal`** (a new arm, retrained here) with **`P3_rolling`**
(canonical, not retrained). They differ only in the graph, which is the whole point of
the pair.

### Run one notebook at a time

Finish this arm, run the cleanup at the bottom **yourself**, shut the kernel down, then
open the next one. Nothing here frees the GPU automatically.

## 0. Free GPU memory right now

In [ ]:
# ---------------------------------------------------------------------------------------
# The ONE thing this notebook does automatically: record how much GPU memory is free right
# now, so the cleanup at the bottom has a number to compare against. Leave this output
# visible, or write the number down.
# ---------------------------------------------------------------------------------------
import sys
from pathlib import Path

PV2 = Path.cwd() if Path.cwd().name == "protocol_v2" else Path.cwd().parent
assert (PV2 / "config.py").exists(), f"run this from protocol_v2/notebooks/, not {Path.cwd()}"

# ORDER MATTERS. Both protocol_v2/ and src/ contain a module called config.py — src's is
# the Sparkov preprocessing config, ours is the protocol-v2 settings. Whichever directory
# comes first on sys.path wins, so protocol_v2 is inserted LAST and therefore ends up
# FIRST. Getting this backwards makes `import config` silently return the wrong module and
# fail two lines later with a confusing AttributeError, so the assertion below names it.
for d in (str(PV2.parent / "src"), str(PV2.parent), str(PV2)):
    while d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)

import torch
import config as CFG
assert Path(CFG.__file__).parent == PV2, (
    f"`import config` picked up {CFG.__file__} instead of {PV2/'config.py'}. "
    f"src/config.py is shadowing it — restart the kernel and run this cell first.")

DEV = CFG.DEVICE
_free_start, _total = torch.cuda.mem_get_info(torch.device(DEV))
print(f"free at start: {_free_start/1e9:.1f} GB of {_total/1e9:.1f} GB on {DEV}")

## 1. Setup, provenance, and the P3_rolling lock

In [ ]:
# --- 1. setup, provenance, and the P3_rolling lock ---------------------------------------
import warnings; warnings.filterwarnings("ignore")
import json, time
import numpy as np, pandas as pd

import guards, provenance as prov, runner, splits as SP
import assembly_settings as ASET

ARM = "P1_causal"

# (a) the canonical P3_rolling directory, hashed BEFORE anything runs. Read only — this
#     notebook does not chmod it, move it, or write to it in any way. See the note in
#     guards.py about why the permissions are deliberately left alone.
lock_before = guards.assert_canonical_unchanged()

# (b) which bytes of source this run is made of. The canonical run recorded
#     git_commit: null because the project had no repository then, so these hashes are
#     what pin the code state for that run; for this one, the commit is recorded too.
src_hashes = prov.print_source_hashes()

# (c) the environment, and — the part the plan was missing — the COMPARISON against the
#     environment the canonical arm was trained in. Differences are printed, not hidden.
env = prov.environment_manifest(runner.find_prepared() / "sparkov_clean_v1.csv")
env_cmp = prov.compare_environment(env)
assert not env_cmp["fatal"], "dataset hash differs from the canonical run — stop"

# (d) freeze the package list once, before arm 1
prov.freeze_requirements()

# (e) freeze the ANALYSIS before any training. Write-once: the first arm notebook to run
#     creates analysis_lock.json, the rest verify against it, and A5 refuses to assemble
#     if either analysis file has moved since. Both files are locked — assembly_settings.py
#     declares the rules and assemble.py implements them.
prov.write_analysis_lock()
ASET.print_frozen()

## 2. Data and fold sets

In [ ]:
# --- 2. load the data, build or load the fold sets ---------------------------------------
# ensure_splits() generates the splits on the first notebook that runs and SAVES them with
# their row indices. Every later notebook loads those same bytes and verifies the hashes.
# That is what makes P1_causal and P1_leaky a matched pair across separate processes.
B = runner.load_frame()
df = B["df"]
sp = B["splits"]
week_of_row = B["week_of_row"]

print()
print(f"slice        {len(df):,} rows, {len(B['num'])} features, fraud "
      f"{df[B['cfg'].target_col].mean()*100:.4f}%")
print(f"P1 split     pair sha256 {sp['p1_pair_sha256_recomputed']}")
print("             ^-- RECOMPUTED from the arrays just loaded off disk, not read out of")
print("                 the manifest. A2 (P1_causal) and A3 (P1_leaky) must print exactly")
print("                 this string. If they do, both arms trained on the same rows.")

## 3. Configuration

In [ ]:
# --- 3. the configuration, read from rerun_canonical.py so it cannot drift ----------------
# Nothing below is restated in this notebook. config.py imports EXPECTED_TRAIN and
# EXPECTED_MODEL from rerun_canonical.py, and asserts epochs == 200 at import time.
CFG.print_summary()

tcfg, mcfg, gcfg = runner.configs_for(ARM, B)
assert tcfg.epochs == 200, "not the canonical budget — stop"
print()
print(f"resolved for {ARM}: epochs={tcfg.epochs} lr={tcfg.lr} seed={tcfg.seed} "
      f"batch={tcfg.batch_size} swa={tcfg.use_swa} device={tcfg.device}")
builder, assert_causality = runner.edge_builder_for(ARM)
print(f"edge_builder={'build_acausal_edges' if builder else 'None -> build_temporal_edges'}"
      f"   assert_causality={assert_causality}")

## 4. Preflight — stop if anything is wrong

In [ ]:
# --- 4. preflight. Stop here if anything is wrong ----------------------------------------
# Fixing a setting costs minutes now and up to ten hours later.
graph_rec = runner.graph_facts(B, "causal")
print()
pre_rec = runner.preflight(ARM, B)

## 5. Train

In [ ]:
# --- 5. train ----------------------------------------------------------------------------
# The single guarded call. runner.train_arm's first statement is guards.assert_trainable,
# and it is the only place in protocol_v2 that calls run_date_gnn_fold.
res, hours = runner.train_arm(ARM, B)
res

## 6. Save

In [ ]:
# --- 6. save, and re-check the lock ------------------------------------------------------
summary = runner.finalise(ARM, res, hours, B, preflight_rec=pre_rec,
                          graph_rec=graph_rec)
print()
lock_after = guards.assert_canonical_unchanged()
assert lock_after["combined_sha256"] == lock_before["combined_sha256"]
print("  canonical P3_rolling is byte-for-byte what it was when this notebook started")
print()
print(json.dumps(summary, indent=1))

---

# GPU cleanup — available on demand. **Nothing here runs by itself.**

The arm is finished and its results are saved. Both cells below are left **unexecuted**.
Neither will do anything until you run it. Nothing in this notebook frees memory
automatically, so the model and the graph are still in memory and you can still inspect
them.

### Step 1 — look at the memory. Run cell **CHECK** below.

It only *reads* the card. It frees nothing, deletes nothing, changes nothing. It prints
how much GPU memory is free now against the number the very first cell printed before
this arm started, so you can see how much this notebook is still holding.

### Step 2 — decide.

| what CHECK tells you | what to do |
|---|---|
| free memory is close to the start number | nothing held. Shut the kernel and move on. |
| this notebook is holding a large amount, and the next arm needs it | run **CLEAN** below |
| you still want to inspect `res`, `B` or the model | do that first — CLEAN deletes them |

### Step 3 — if you decided to clean, run cell **CLEAN** below.

It deletes this notebook's big objects, empties the CUDA cache, and prints free memory
before and after so you can see what it actually recovered. It ends there: no `assert`,
no automatic action, nothing that decides for you. You can run CHECK again afterwards.

### Step 4 — shut the kernel down. Kernel → Shut Down Kernel.

Do this whether or not you ran CLEAN. Killing the process is the only real guarantee that
the card is clear; a free-memory reading is not. **Only then open the next notebook.**

One trade-off, stated plainly: if the card is still full when the next arm starts, that
arm fails on a memory error partway through — an hour on A2 or A3, up to nine on A4. The
preflight in step 4 of the next notebook checks free memory against the projected peak and
stops before training if there is not enough, so this should be caught rather than
discovered at hour six.

**Next:** `A3_P1_leaky.ipynb`, which must load the *same* P1 split. Compare the `pair sha256` printed in step 2 of both notebooks.

In [ ]:
# ============================================================================== CHECK
# READ-ONLY. Frees nothing, deletes nothing. Run it as often as you like.
# Shows what this notebook is currently holding on the GPU.
# ==============================================================================
import torch

_free_now, _total = torch.cuda.mem_get_info(torch.device(DEV))
_held = (_free_start - _free_now) / 1e9

print(f"device            {DEV}")
print(f"free at start     {_free_start/1e9:6.1f} GB   (printed by the first cell)")
print(f"free now          {_free_now/1e9:6.1f} GB")
print(f"total on card     {_total/1e9:6.1f} GB")
print(f"held by this run  {_held:6.2f} GB")
print()
if _held < 0.5:
    print("Under 0.5 GB is held. Nothing to clean. Shut the kernel down and move on.")
else:
    print(f"This notebook is holding {_held:.1f} GB.")
    print("Run the CLEAN cell below if the next arm needs it, or just shut the kernel")
    print("down, which releases everything regardless.")
print()
print("Objects still alive:",
      [n for n in ["model", "opt", "sched", "res", "res_smoke", "csr", "x_np", "df",
                   "ei_np", "ea_np", "B", "sp", "week_of_row"] if n in globals()])

In [ ]:
# ============================================================================== CLEAN
# DESTRUCTIVE, and only if you ran it. Deletes this notebook's big objects and empties
# the CUDA cache. Results on disk are untouched. Run CHECK again afterwards to see the
# effect. Nothing is asserted and nothing is decided for you.
# ==============================================================================
import gc, torch

_before, _total = torch.cuda.mem_get_info(torch.device(DEV))

_deleted = []
for _name in ["model", "opt", "sched", "res", "res_smoke", "csr", "x_np", "df",
              "ei_np", "ea_np", "B", "sp", "week_of_row"]:
    if _name in globals():
        del globals()[_name]
        _deleted.append(_name)

gc.collect()
torch.cuda.empty_cache()

_after, _total = torch.cuda.mem_get_info(torch.device(DEV))
print(f"deleted           {_deleted or 'nothing — already cleared'}")
print(f"free before CLEAN {_before/1e9:6.1f} GB")
print(f"free after  CLEAN {_after/1e9:6.1f} GB")
print(f"recovered         {(_after - _before)/1e9:6.2f} GB")
print(f"free at start     {_free_start/1e9:6.1f} GB")
print(f"still held        {(_free_start - _after)/1e9:6.2f} GB")
print()
print("Under 0.5 GB still held is fine. Anything more means the memory is not coming")
print("back in this process — shut the kernel down, which always clears it.")